In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import seaborn as sns
import pyarrow
DATA_PATH = "./data/"

#### Convert csv to parquet

In [6]:
#data_converter(f"{DATA_PATH}Traffic_Crashes_-_Crashes_20260702.csv", f"{DATA_PATH}traffic_crashes_260702.parquet")

In [2]:
df = pd.read_csv("data/Traffic_Crashes_-_Crashes_20260702.csv")
#df_people = pd.read_csv("data/Traffic_Crashes_-_People_20260702.csv")
#df_vehicle = pd.read_csv("data/Traffic_Crashes_-_Vehicles_20260702.csv")

/tmp/ipykernel_164130/2723038170.py:1: DtypeWarning: Columns (0: LANE_CNT) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/Traffic_Crashes_-_Crashes_20260702.csv")


In [3]:
df.head()

,CRASH_RECORD_ID,CRASH_DATE_EST_I,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,...,INJURIES_NON_INCAPACITATING,INJURIES_REPORTED_NOT_EVIDENT,INJURIES_NO_INDICATION,INJURIES_UNKNOWN,CRASH_HOUR,CRASH_DAY_OF_WEEK,IDOT_CONTROL_NO,LATITUDE,LONGITUDE,LOCATION
0,006bc88aa00209dba00e7f4e77939f4cac6d1918b26e0f...,NaN,07/28/2020 09:37:00 PM,25,NO CONTROLS,NO CONTROLS,CLEAR,DARKNESS,FIXED OBJECT,NOT DIVIDED,...,1.0,0.0,3.0,0.0,21,3,X001971047,NaN,NaN,NaN
1,8bd0613f9b39d21c255f4c0c6964a95418c448fe25015b...,NaN,08/12/2020 06:49:00 PM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,DUSK,SIDESWIPE SAME DIRECTION,FOUR WAY,...,0.0,0.0,2.0,0.0,18,4,X001984876,NaN,NaN,NaN
2,54bb43f31434a799998791acfc9020eb0a09401332a08e...,NaN,08/09/2020 01:55:00 AM,25,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",HEAD ON,NOT DIVIDED,...,5.0,0.0,0.0,0.0,1,1,X001981337,NaN,NaN,NaN
3,3d8f4ecfbdce63facb0497b9b00b4538e0f75fe7de3ad8...,NaN,07/27/2020 09:30:00 AM,30,NO CONTROLS,NO CONTROLS,RAIN,DAYLIGHT,PARKED MOTOR VEHICLE,ONE-WAY,...,0.0,0.0,1.0,0.0,9,2,NaN,41.797460,-87.620768,POINT (-87.62076816217 41.797459871568)
4,b454dcc68b7f1cc9dbae0e8f8caa438d184ded31584554...,NaN,08/10/2020 04:30:00 PM,30,NO CONTROLS,OTHER,RAIN,DAYLIGHT,OTHER NONCOLLISION,OTHER,...,NaN,NaN,NaN,NaN,16,2,X001983368,41.790535,-87.800059,POINT (-87.800059225607 41.790535405928)


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1070335 entries, 0 to 1070334
Data columns (total 49 columns):
 #   Column                         Non-Null Count    Dtype  
---  ------                         --------------    -----  
 0   CRASH_RECORD_ID                1070335 non-null  str    
 1   CRASH_DATE_EST_I               77380 non-null    str    
 2   CRASH_DATE                     1070335 non-null  str    
 3   POSTED_SPEED_LIMIT             1070335 non-null  int64  
 4   TRAFFIC_CONTROL_DEVICE         1070335 non-null  str    
 5   DEVICE_CONDITION               1070335 non-null  str    
 6   WEATHER_CONDITION              1070335 non-null  str    
 7   LIGHTING_CONDITION             1070335 non-null  str    
 8   FIRST_CRASH_TYPE               1070335 non-null  str    
 9   TRAFFICWAY_TYPE                1070335 non-null  str    
 10  LANE_CNT                       199044 non-null   object 
 11  ALIGNMENT                      1070335 non-null  str    
 12  ROADWAY_SURFACE_COND     

In [10]:
#Function that returns a report of nulls given a dataframe in %
def null_pct_report(X):
    return X.isnull().sum()/len(X)*100

In [11]:
#Report of nulls
nulls_report = null_pct_report(df)

In [12]:
#pd.Series of features with higher than 70% of nulls
too_many_nulls = nulls_report[nulls_report > 70]

In [13]:
#Extract dtypes from the features with more than 70% of nulls
df[too_many_nulls.index.tolist()].dtypes

CRASH_DATE_EST_I             str
LANE_CNT                  object
INTERSECTION_RELATED_I       str
NOT_RIGHT_OF_WAY_I           str
PHOTOS_TAKEN_I               str
STATEMENTS_TAKEN_I           str
DOORING_I                    str
WORK_ZONE_I                  str
WORK_ZONE_TYPE               str
WORKERS_PRESENT_I            str
dtype: object

In [14]:
#Inspection of too many nulls
for col in too_many_nulls.index:
    print(f"{col}:\n{df[col].value_counts()}\n")

CRASH_DATE_EST_I:
CRASH_DATE_EST_I
Y    67598
N     9782
Name: count, dtype: int64

LANE_CNT:
LANE_CNT
2.0          85582
4.0          46363
1.0          30511
3.0           8117
0.0           7543
2             5593
6.0           4236
4             3229
1             2043
5.0           1808
8.0           1787
3              562
0              489
6              267
7.0            173
10.0           153
5              132
8              121
99.0           100
9.0             63
11.0            29
12.0            28
20.0            14
22.0            12
7               11
10               9
99               8
16.0             7
15.0             7
14.0             5
30.0             5
40.0             4
9                3
60.0             3
21.0             3
25.0             2
100.0            2
28.0             1
41               1
20               1
433,634          1
35               1
1,191,625        1
12               1
19.0             1
400.0            1
902.0            1
299,

### Treating nulls

In [15]:
#Treat nulls

mapping = {"Y":1, "N":0}
#Selecting string binary columns to map to integers
inpute = [col for col in too_many_nulls.index if df[col].nunique() < 3]

# After selecting binary columns, we will replace them for integers and fill nulls with 0. 
# I know it's not the best approach for filling in nulls in this case.
df[inpute] = df[inpute].replace(mapping).fillna(0).astype(int)

#Check what is left to impute on the binary strings
still_null = list(set(inpute) ^ set(too_many_nulls.index))

#Inpute LANE_CNT


In [16]:
df["NOT_RIGHT_OF_WAY_I"].value_counts()

NOT_RIGHT_OF_WAY_I
0    1027059
1      43276
Name: count, dtype: int64

In [17]:
df["CRASH_DATE_EST_I"].value_counts()

CRASH_DATE_EST_I
0    1002737
1      67598
Name: count, dtype: int64

In [18]:
df[["REPORT_TYPE", "CRASH_DATE_EST_I", "DATE_POLICE_NOTIFIED", "CRASH_DATE"]]

,REPORT_TYPE,CRASH_DATE_EST_I,DATE_POLICE_NOTIFIED,CRASH_DATE
0,ON SCENE,0,07/28/2020 10:38:00 PM,07/28/2020 09:37:00 PM
1,NOT ON SCENE (DESK REPORT),0,08/12/2020 09:00:00 PM,08/12/2020 06:49:00 PM
2,ON SCENE,0,08/09/2020 01:55:00 AM,08/09/2020 01:55:00 AM
3,ON SCENE,0,07/27/2020 09:30:00 AM,07/27/2020 09:30:00 AM
4,NOT ON SCENE (DESK REPORT),0,08/11/2020 01:40:00 PM,08/10/2020 04:30:00 PM
...,...,...,...,...
1070330,NOT ON SCENE (DESK REPORT),0,06/20/2026 10:30:00 PM,06/20/2026 09:45:00 PM
1070331,NOT ON SCENE (DESK REPORT),0,06/23/2026 05:50:00 PM,06/23/2026 02:19:00 PM
1070332,ON SCENE,0,06/23/2026 12:23:00 PM,06/23/2026 12:15:00 PM
1070333,NOT ON SCENE (DESK REPORT),0,06/23/2026 08:45:00 PM,06/20/2026 12:45:00 PM


In [19]:
pd.crosstab(df['REPORT_TYPE'], df['CRASH_DATE_EST_I'])

CRASH_DATE_EST_I,0,1
REPORT_TYPE,,
AMENDED,221,19
NOT ON SCENE (DESK REPORT),540729,35982
ON SCENE,427530,29630


In [20]:
df[(df["REPORT_TYPE"] == "ON SCENE" )& (df["CRASH_DATE_EST_I"] == 1)]["MOST_SEVERE_INJURY"].value_counts()

MOST_SEVERE_INJURY
NO INDICATION OF INJURY     24642
NONINCAPACITATING INJURY     2720
REPORTED, NOT EVIDENT        1178
INCAPACITATING INJURY         725
FATAL                          61
Name: count, dtype: int64

In [21]:
still_null

['LANE_CNT', 'WORK_ZONE_TYPE']

In [22]:
#LANE_CNT
df["LANE_CNT"].value_counts()

LANE_CNT
2.0          85582
4.0          46363
1.0          30511
3.0           8117
0.0           7543
2             5593
6.0           4236
4             3229
1             2043
5.0           1808
8.0           1787
3              562
0              489
6              267
7.0            173
10.0           153
5              132
8              121
99.0           100
9.0             63
11.0            29
12.0            28
20.0            14
22.0            12
7               11
10               9
99               8
16.0             7
15.0             7
14.0             5
30.0             5
40.0             4
9                3
60.0             3
21.0             3
25.0             2
100.0            2
28.0             1
41               1
20               1
433,634          1
35               1
1,191,625        1
12               1
19.0             1
400.0            1
902.0            1
299,679          1
17.0             1
45.0             1
13.0             1
11               1
218

In [23]:
df["LANE_CNT_CLEAN"] = pd.to_numeric(df["LANE_CNT"].astype(str).str.replace(',', ''), errors='coerce')

In [24]:
df.loc[df["LANE_CNT_CLEAN"] > 16, "LANE_CNT_CLEAN"] = np.nan
df["LANE_CNT_CLEAN"] = df["LANE_CNT_CLEAN"].fillna(-1).astype(int)
df.drop(columns="LANE_CNT", inplace=True)

In [25]:
df.rename(columns={"LANE_CNT_CLEAN":"LANE_CNT"},inplace=True)

In [26]:
df["WORK_ZONE_TYPE"].value_counts(dropna=False)

WORK_ZONE_TYPE
NaN             1065974
CONSTRUCTION       3025
UNKNOWN             620
MAINTENANCE         450
UTILITY             266
Name: count, dtype: int64

In [27]:
#Moving all NaN to UNKNOWN
df["WORK_ZONE_TYPE"] = df["WORK_ZONE_TYPE"].fillna("UNKNOWN")

In [28]:
serie = null_pct_report(df)
serie[serie > 0]

REPORT_TYPE                       3.384361
HIT_AND_RUN_I                    68.611229
STREET_DIRECTION                  0.000374
STREET_NAME                       0.000093
BEAT_OF_OCCURRENCE                0.000467
MOST_SEVERE_INJURY                0.217222
INJURIES_TOTAL                    0.215820
INJURIES_FATAL                    0.215820
INJURIES_INCAPACITATING           0.215820
INJURIES_NON_INCAPACITATING       0.215820
INJURIES_REPORTED_NOT_EVIDENT     0.215820
INJURIES_NO_INDICATION            0.215820
INJURIES_UNKNOWN                  0.215820
IDOT_CONTROL_NO                   0.262254
LATITUDE                          0.774430
LONGITUDE                         0.774430
LOCATION                          0.774430
dtype: float64

In [29]:
df["REPORT_TYPE"].value_counts(dropna=False)

REPORT_TYPE
NOT ON SCENE (DESK REPORT)    576711
ON SCENE                      457160
NaN                            36224
AMENDED                          240
Name: count, dtype: int64

In [30]:
not_important = serie[(serie > 0 ) & (serie < 1)].index
not_important

Index(['STREET_DIRECTION', 'STREET_NAME', 'BEAT_OF_OCCURRENCE',
       'MOST_SEVERE_INJURY', 'INJURIES_TOTAL', 'INJURIES_FATAL',
       'INJURIES_INCAPACITATING', 'INJURIES_NON_INCAPACITATING',
       'INJURIES_REPORTED_NOT_EVIDENT', 'INJURIES_NO_INDICATION',
       'INJURIES_UNKNOWN', 'IDOT_CONTROL_NO', 'LATITUDE', 'LONGITUDE',
       'LOCATION'],
      dtype='str')

In [31]:
df.dropna(axis=0, subset=not_important, inplace=True, ignore_index=True)

In [32]:
serie = null_pct_report(df)
rest_nulls = (serie[serie > 0]).index

In [33]:
for col in rest_nulls:
    print(df[col].value_counts(dropna=False), "\n")

REPORT_TYPE
NOT ON SCENE (DESK REPORT)    572290
ON SCENE                      448499
NaN                            35962
AMENDED                          237
Name: count, dtype: int64 

HIT_AND_RUN_I
NaN    724430
Y      318359
N       14199
Name: count, dtype: int64 



In [34]:
#Imputation REPORT_TYPE
df[["REPORT_TYPE","HIT_AND_RUN_I"]] = df[["REPORT_TYPE", "HIT_AND_RUN_I"]].fillna("UNKNOWN")

In [35]:
print(f"Number of nulls: {null_pct_report(df).sum()}")

Number of nulls: 0.0


In [36]:
df["TRAFFIC_CONTROL_DEVICE"].value_counts()

TRAFFIC_CONTROL_DEVICE
NO CONTROLS                 592070
TRAFFIC SIGNAL              293701
STOP SIGN/FLASHER           105582
UNKNOWN                      50013
OTHER                         7084
YIELD                         1628
OTHER REG. SIGN               1215
LANE USE MARKING              1170
PEDESTRIAN CROSSING SIGN       881
OTHER WARNING SIGN             773
RAILROAD CROSSING GATE         675
FLASHING CONTROL SIGNAL        479
SCHOOL ZONE                    435
DELINEATORS                    360
POLICE/FLAGMAN                 345
RR CROSSING SIGN               244
OTHER RAILROAD CROSSING        216
NO PASSING                      76
BICYCLE CROSSING SIGN           41
Name: count, dtype: int64

In [37]:
df["CRASH_TYPE"].value_counts()

CRASH_TYPE
NO INJURY / DRIVE AWAY              770498
INJURY AND / OR TOW DUE TO CRASH    286490
Name: count, dtype: int64

In [38]:
df["CRASH_DATE"] = pd.to_datetime(df["CRASH_DATE"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
df["DATE_POLICE_NOTIFIED"] = pd.to_datetime(df["DATE_POLICE_NOTIFIED"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")

#We should drop some columns which redundant having CRASH_DATE. Those are CRASH_MONTH, CRASH_DAY_OF_WEEK
df = df.drop(columns=["CRASH_MONTH","CRASH_DAY_OF_WEEK"])

In [39]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1056988 entries, 0 to 1056987
Data columns (total 47 columns):
 #   Column                         Non-Null Count    Dtype         
---  ------                         --------------    -----         
 0   CRASH_RECORD_ID                1056988 non-null  str           
 1   CRASH_DATE_EST_I               1056988 non-null  int64         
 2   CRASH_DATE                     1056988 non-null  datetime64[us]
 3   POSTED_SPEED_LIMIT             1056988 non-null  int64         
 4   TRAFFIC_CONTROL_DEVICE         1056988 non-null  str           
 5   DEVICE_CONDITION               1056988 non-null  str           
 6   WEATHER_CONDITION              1056988 non-null  str           
 7   LIGHTING_CONDITION             1056988 non-null  str           
 8   FIRST_CRASH_TYPE               1056988 non-null  str           
 9   TRAFFICWAY_TYPE                1056988 non-null  str           
 10  ALIGNMENT                      1056988 non-null  str           
 

### Feature Engineering

In [40]:
df_test = df["STREET_NAME"].str.rsplit(" ", n=1, expand=True)
df.rename(columns={"STREET_NAME":"FULL_STREET_NAME"}, inplace=True)

df["STREET_NAME"] = df_test[0]
df["TYPE_STREET"] = df_test[1]



In [41]:
mapping_type_street = {
    "AVE": "AVE", "AVE.": "AVE", "AVENUE": "AVE",
    "ST": "ST", "STREET": "ST",
    "RD": "RD",
    "BLVD": "BLVD", "BLVD.": "BLVD",
    "DR": "DR", "DRIVE": "DR",
    "PL": "PL", "PKWY": "PKWY", "CT": "CT", "HWY": "HWY", 
    "TER": "TER", "EXPY": "EXPY", "PLZ": "PLZ", "WAY": "WAY", "LN": "LN",
    "NB": "NB", "SB": "SB", "IB": "IB", "OB": "OB", 
    "RAMP": "RAMP", "NB-RAMP": "RAMP", "OVERPASS": "OVERPASS", "BUSWAY": "BUSWAY"
}

In [42]:
df["TYPE_STREET_CLEAN"] = df["TYPE_STREET"].map(mapping_type_street).fillna("OTHER")

In [43]:
df.drop(columns=["TYPE_STREET"], inplace=True)

In [44]:
df.rename(columns={"TYPE_STREET_CLEAN":"TYPE_STREET"}, inplace=True)

## CREATION OF SQL DB


### Facts

#### fact_crash

In [45]:
cols_fact_crash = (["CRASH_RECORD_ID", "POSTED_SPEED_LIMIT", "CRASH_TYPE",
                    "DATE_POLICE_NOTIFIED","CRASH_DATE_EST_I",
                    "DAMAGE", "WORK_ZONE_I", "WORK_ZONE_TYPE", "WORKERS_PRESENT_I", 
                    "NUM_UNITS","MOST_SEVERE_INJURY", "INJURIES_TOTAL","INJURIES_FATAL",
                    "INJURIES_INCAPACITATING", "INJURIES_NON_INCAPACITATING", "INJURIES_REPORTED_NOT_EVIDENT",
                    "INJURIES_NO_INDICATION", "INJURIES_UNKNOWN"])

### Dimensions

#### dim_location

In [46]:
cols_location = ["LATITUDE","LONGITUDE","LOCATION","STREET_NO","STREET_DIRECTION","FULL_STREET_NAME","STREET_NAME","TYPE_STREET","BEAT_OF_OCCURRENCE"]

#What to do with WORK_ZONE_I and WORK_ZONE_TYPE

#### dim_time

In [47]:
cols_time = ["CRASH_DATE"]

#### dim_road_carachteristics

In [48]:
cols_road_crtcs = (
    ["TRAFFICWAY_TYPE","LANE_CNT","ALIGNMENT", "ROADWAY_SURFACE_COND",
     "ROAD_DEFECT", "INTERSECTION_RELATED_I", "NOT_RIGHT_OF_WAY_I","TRAFFIC_CONTROL_DEVICE",
     "DEVICE_CONDITION"])

#### dim_ambiental_conditions

In [49]:
cols_amb_cnd = ["WEATHER_CONDITION", "LIGHTING_CONDITION"]

#### dim_administrative_report

In [50]:
cols_admin_report = ["IDOT_CONTROL_NO", "REPORT_TYPE", "PHOTOS_TAKEN_I", "STATEMENTS_TAKEN_I"]

#### dim_causes_types_crash

In [51]:
cols_causes_types = ["PRIM_CONTRIBUTORY_CAUSE", "SEC_CONTRIBUTORY_CAUSE", "FIRST_CRASH_TYPE", "DOORING_I", "HIT_AND_RUN_I"]

### Creating df for each dimension and fact

#### dim_location

In [52]:
dim_location = df[cols_location].drop_duplicates().reset_index(drop=True)
dim_location["location_id"] = dim_location.index + 1
dim_location = dim_location[["location_id"] + cols_location]

#### dim_time

In [53]:
#cols_time = ["CRASH_DATE","DATE_POLICE_NOTIFIED","CRASH_MONTH","CRASH_HOUR","CRASH_DAY_OF_WEEK", "CRASH_DATE_EST_I"]
dim_time = pd.DataFrame({"complete_date": df['CRASH_DATE'].dt.normalize()}).drop_duplicates().reset_index(drop=True)
dim_time["year"] = dim_time["complete_date"].dt.year
dim_time["month"] = dim_time["complete_date"].dt.month
dim_time["day"] = dim_time["complete_date"].dt.day
dim_time["day_of_week"] = dim_time["complete_date"].dt.dayofweek
dim_time["is_weekend"] = dim_time["day_of_week"].isin([5,6]).astype(int)

dim_time["time_id"] = (dim_time["year"] * 10000 + dim_time["month"] * 100 + dim_time["day"])

#### dim_ambiental_conditions

In [54]:
dim_amb_cnd = df[cols_amb_cnd].drop_duplicates().reset_index(drop=True)
dim_amb_cnd["amb_cnd_id"] = dim_amb_cnd.index + 1
dim_amb_cnd = dim_amb_cnd[["amb_cnd_id"] + cols_amb_cnd]

#### dim_road_crtcs

In [55]:
dim_road_crtcs = df[cols_road_crtcs].drop_duplicates().reset_index(drop=True)
dim_road_crtcs["road_crtcs_id"] = dim_road_crtcs.index + 1
dim_road_crtcs = dim_road_crtcs[["road_crtcs_id"] + cols_road_crtcs]

#### dim_causes_types_crash

In [56]:
dim_causes_types = df[cols_causes_types].drop_duplicates().reset_index(drop=True)
dim_causes_types["causes_types_id"] = dim_causes_types.index + 1
dim_causes_types = dim_causes_types[["causes_types_id"] + cols_causes_types]

#### dim_administrative_report

In [57]:
dim_admin_report = df[cols_admin_report].drop_duplicates().reset_index(drop=True)
dim_admin_report["admin_report_id"] = dim_admin_report.index + 1
dim_admin_report = dim_admin_report[["admin_report_id"] + cols_admin_report]

#### fact_crash

In [58]:
fact_crash = df[cols_fact_crash].copy()

fact_crash["time_id"] = df["CRASH_DATE"].dt.strftime("%Y%m%d").astype(int)
fact_crash["CRASH_HOUR"] = df["CRASH_DATE"].dt.hour.astype(int)
fact_crash["CRASH_MINUTE"] = df["CRASH_DATE"].dt.minute.astype(int)

#Merge location
fact_crash = fact_crash.merge(
    df[["CRASH_RECORD_ID"] + cols_location], on="CRASH_RECORD_ID", how="left"
)
fact_crash = fact_crash.merge(dim_location, on=cols_location, how="left")
fact_crash = fact_crash.drop(columns=cols_location)

#Merge ambiental_conditions
fact_crash = fact_crash.merge(
    df[["CRASH_RECORD_ID"] + cols_amb_cnd], on="CRASH_RECORD_ID", how="left"
)
fact_crash = fact_crash.merge(dim_amb_cnd, on=cols_amb_cnd, how="left")
fact_crash = fact_crash.drop(columns=cols_amb_cnd)

#Merge road characteristics
fact_crash = fact_crash.merge(
    df[["CRASH_RECORD_ID"] + cols_road_crtcs], on="CRASH_RECORD_ID", how="left"
)
fact_crash = fact_crash.merge(dim_road_crtcs, on=cols_road_crtcs, how="left")
fact_crash = fact_crash.drop(columns=cols_road_crtcs)

#Merge causes types crash
fact_crash = fact_crash.merge(
    df[["CRASH_RECORD_ID"] + cols_causes_types], on="CRASH_RECORD_ID", how="left"
)
fact_crash = fact_crash.merge(dim_causes_types, on=cols_causes_types, how="left")
fact_crash = fact_crash.drop(columns=cols_causes_types)

#Merge administrative reports
fact_crash = fact_crash.merge(
    df[["CRASH_RECORD_ID"] + cols_admin_report], on="CRASH_RECORD_ID", how="left"
)
fact_crash = fact_crash.merge(dim_admin_report, on=cols_admin_report, how="left")
fact_crash = fact_crash.drop(columns=cols_admin_report)

### To SQL

In [59]:
import sqlite3

In [61]:
DB_NAME = "chicago_crashes_warehouse.db"

print("Creating connetion...")
conn = sqlite3.connect(DB_NAME)

try:
    #dim_time
    dim_time.set_index("time_id", inplace=True)
    dim_time.to_sql("dim_time", con=conn, if_exists="replace", index=True, index_label="time_id")
    
    #dim_location
    dim_location.set_index("location_id", inplace=True)
    dim_location.to_sql("dim_location", con=conn, if_exists="replace", index=True, index_label="location_id")
    
    #dim_amb_cnd
    dim_amb_cnd.set_index("amb_cnd_id", inplace=True)
    dim_amb_cnd.to_sql("dim_amb_cnd", con=conn, if_exists="replace", index=True, index_label="amb_cnd_id")
    
    #dim_road_crtcs
    dim_road_crtcs.set_index("road_crtcs_id", inplace=True)
    dim_road_crtcs.to_sql("dim_road_crtcs", con=conn, if_exists="replace", index=True, index_label="road_crtcs_id")
    
    #dim_causes_types
    dim_causes_types.set_index("causes_types_id", inplace=True)
    dim_causes_types.to_sql("dim_causes_types", con=conn, if_exists="replace", index=True, index_label="causes_types_id")
    
    #dim_admin_report
    dim_admin_report.set_index("admin_report_id", inplace=True)
    dim_admin_report.to_sql("dim_admin_report", con=conn, if_exists="replace", index=True, index_label="admin_report_id")
    
    
    print("Creating fact_crash table")
    fact_crash.set_index("CRASH_RECORD_ID", inplace=True)
    fact_crash.to_sql("fact_crash", con=conn, if_exists="replace", index=True, index_label="CRASH_RECORD_ID", chunksize=100000)
finally:
    conn.close()
    print("Connection closed")

Creating connetion...
Creating fact_crash table
Connection closed
